# Implementing expert networks

This notebook builds one SwiGLU expert, an independent expert bank, a shared path, and a manually selected routed path. Top-k routing comes later.

In [1]:
import sys
from pathlib import Path

import torch

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "model").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find the repository root")
sys.path.insert(0, str(project_root))
torch.manual_seed(16)

In [2]:
from src.model import DeepSeekConfig, Expert, ExpertBank

expert = Expert(dim=256, hidden_dim=256)
x = torch.randn(2, 16, 256)
y = expert(x)
print("Expert output:", y.shape)
print("Expert parameters:", sum(p.numel() for p in expert.parameters()))
assert y.shape == x.shape

Expert output: torch.Size([2, 16, 256])
Expert parameters: 196608


In [3]:
config = DeepSeekConfig()
bank = ExpertBank(config)
shared = bank.forward_shared(x)
expert_0 = bank.forward_expert(0, x)
expert_3 = bank.forward_expert(3, x)
print("Shared:", shared.shape)
print("Expert 0:", expert_0.shape)
print("Expert 3:", expert_3.shape)
print("Expert outputs equal:", torch.allclose(expert_0, expert_3))

Shared: torch.Size([2, 16, 256])
Expert 0: torch.Size([2, 16, 256])
Expert 3: torch.Size([2, 16, 256])
Expert outputs equal: False


In [4]:
output = (
    shared
    + 0.6 * bank.forward_expert(1, x)
    + 0.4 * bank.forward_expert(4, x)
)
loss = output.square().mean()
loss.backward()

print("Combined output:", output.shape)
print("Shared gradient:", bank.shared_expert.gate_proj.weight.grad is not None)
for index, expert in enumerate(bank.routed_experts):
    has_gradient = expert.gate_proj.weight.grad is not None
    print(f"Expert {index} gradient: {has_gradient}")

Combined output: torch.Size([2, 16, 256])
Shared gradient: True
Expert 0 gradient: False
Expert 1 gradient: True
Expert 2 gradient: False
Expert 3 gradient: False
Expert 4 gradient: True
Expert 5 gradient: False
Expert 6 gradient: False
Expert 7 gradient: False


In [5]:
expert_params = 3 * config.dim * config.moe_inter_dim
total = sum(parameter.numel() for parameter in bank.parameters())
active = sum(parameter.numel() for parameter in bank.shared_expert.parameters()) + 2 * expert_params
print("Total expert-bank parameters:", total)
print("Active expert parameters for shared + two routed experts:", active)

Total expert-bank parameters: 1769472
Active expert parameters for shared + two routed experts: 589824


The selected expert IDs and weights above are fixed manually. The next implementation adds token-dependent top-k routing and dispatch.